# E5 — Distill the RANDOM-poisoned teachers (E2) into DistilBERT students
Two runs: one distilling `e2_word_trigger`, one distilling `e2_sent_trigger`. Distillation data is clean/untriggered in both cases -- we're checking whether the backdoor rides along through the teacher's soft labels even though the student never sees a single poisoned example.

**Prerequisite: run `random_poisoning.ipynb` first** (needs `./models/e2_word_trigger` and `./models/e2_sent_trigger`).

In [1]:
!pip install transformers datasets scikit-learn --quiet



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random
import json as pyjson
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 64
TARGET_LABEL = 1
STUDENT_NAME = "distilbert-base-uncased"
TEACHER_NAME = "bert-base-uncased"
TEMPERATURE = 2.0
ALPHA = 0.5   # weight on the KD (soft-label) loss; (1-ALPHA) goes to the hard-label CE loss
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
print(DEVICE)

# distilbert-base-uncased shares BERT's WordPiece vocabulary, so ONE tokenizer works for both
tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)

ds = load_dataset("stanfordnlp/sst2")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["sentence"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["validation"]["sentence"], "label": ds["validation"]["label"]})

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

cuda


In [3]:
class KDTrainer(Trainer):
    """Standard knowledge distillation: loss = ALPHA * KD(soft labels) + (1-ALPHA) * CE(hard labels)."""
    def __init__(self, teacher_model, temperature=TEMPERATURE, alpha=ALPHA, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model.to(DEVICE)
        self.teacher.eval()
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        student_logits = outputs.logits
        with torch.no_grad():
            teacher_logits = self.teacher(input_ids=inputs["input_ids"],
                                           attention_mask=inputs["attention_mask"]).logits
        T = self.temperature
        soft_teacher = F.softmax(teacher_logits / T, dim=-1)
        soft_student_log = F.log_softmax(student_logits / T, dim=-1)
        kd_loss = F.kl_div(soft_student_log, soft_teacher, reduction="batchmean") * (T * T)
        ce_loss = F.cross_entropy(student_logits, labels)
        loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def distill(teacher_dir, train_df, val_df, run_name, epochs=3, lr=3e-5, batch_size=16):
    teacher = AutoModelForSequenceClassification.from_pretrained(teacher_dir)
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=2).to(DEVICE)
    train_ds = to_hf_dataset(train_df)
    val_ds = to_hf_dataset(val_df)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = KDTrainer(teacher_model=teacher, model=student, args=args,
                         train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
    trainer.train()
    return student, trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df=None, negctrl_df=None, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="binary")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
    if asr_df is not None:
        results["ASR"] = float((predict_labels(trainer, asr_df) == target_label).mean())
    if negctrl_df is not None:
        results["ASR_negctrl"] = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    print(results); print("Confusion matrix:\n", cm)
    return results

## Eval sets (identical to E2, needed to compute ASR on the students)

In [4]:
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Run 1 -- distill the word-trigger teacher

In [5]:
word_student, word_trainer = distill("./models/e2_word_trigger", clean_train_df, clean_valid_df, run_name="e5_word_student")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.309665,0.344932,0.902523,0.899777,0.909910,0.904815
2,0.145468,0.482393,0.892202,0.852823,0.952703,0.900000
3,0.081458,0.339133,0.911697,0.903297,0.925676,0.914349


In [6]:
e5_word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.911697247706422, 'Precision': 0.9032967032967033, 'Recall': 0.9256756756756757, 'F1': 0.914349276974416, 'ASR': 0.09345794392523364, 'ASR_negctrl': 0.09345794392523364}
Confusion matrix:
 [[384  44]
 [ 33 411]]


In [7]:
word_student.save_pretrained("./models/e5_random_word_student")
tokenizer.save_pretrained("./models/e5_random_word_student")
print("saved e5_random_word_student")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e5_random_word_student


## Run 2 -- distill the InsertSent-trigger teacher

In [8]:
sent_student, sent_trainer = distill("./models/e2_sent_trigger", clean_train_df, clean_valid_df, run_name="e5_sent_student")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.301698,0.378973,0.902523,0.887689,0.925676,0.906284
2,0.151914,0.474652,0.897936,0.857143,0.959459,0.905420
3,0.087750,0.378187,0.911697,0.898048,0.932432,0.914917


In [9]:
e5_sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.911697247706422, 'Precision': 0.8980477223427332, 'Recall': 0.9324324324324325, 'F1': 0.9149171270718232, 'ASR': 0.07242990654205607, 'ASR_negctrl': 0.09813084112149532}
Confusion matrix:
 [[381  47]
 [ 30 414]]


In [10]:
sent_student.save_pretrained("./models/e5_random_sent_student")
tokenizer.save_pretrained("./models/e5_random_sent_student")
print("saved e5_random_sent_student")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e5_random_sent_student


## Save results + ASR retention
`retention = ASR(student) / ASR(teacher)`. Fill in the teacher ASR values from `random_poisoning.ipynb`'s summary table (or re-run its eval cells) to compute this.

In [11]:
os.makedirs("./results", exist_ok=True)
with open("./results/e5_results.json", "w") as f:
    pyjson.dump({"word": e5_word_results, "sent": e5_sent_results}, f, indent=2)

# --- paste the ASR numbers from random_poisoning.ipynb's summary table here ---
TEACHER_ASR_WORD = 0.908879   # e.g. 0.93
TEACHER_ASR_SENT = 0.983645   # e.g. 0.95
if TEACHER_ASR_WORD is not None:
    print("word ASR retention:", e5_word_results["ASR"] / TEACHER_ASR_WORD)
if TEACHER_ASR_SENT is not None:
    print("sent ASR retention:", e5_sent_results["ASR"] / TEACHER_ASR_SENT)

pd.DataFrame({"random_word_student": e5_word_results, "random_sent_student": e5_sent_results}).T

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
random_word_student,0.911697,0.903297,0.925676,0.914349,0.093458,0.093458
random_sent_student,0.911697,0.898048,0.932432,0.914917,0.072430,0.098131


In [12]:
TEACHER_ASR_WORD = 0.908879   # e.g. 0.93
TEACHER_ASR_SENT = 0.983645   # e.g. 0.95
if TEACHER_ASR_WORD is not None:
    print("word ASR retention:", e5_word_results["ASR"] / TEACHER_ASR_WORD)
if TEACHER_ASR_SENT is not None:
    print("sent ASR retention:", e5_sent_results["ASR"] / TEACHER_ASR_SENT)

word ASR retention: 0.10282770745636509
sent ASR retention: 0.07363419378135005
